In [ ]:
# transformers v5: restore head masks + fix attention mask API for IDEA Grounding DINO
from pathlib import Path
import sys

_pr = Path(".").resolve()
if str(_pr) not in sys.path:
    sys.path.insert(0, str(_pr))
import groundingdino_transformers_compat
groundingdino_transformers_compat.apply()

import os

# Antes de cargar el modelo: fija la GPU (solo afecta si se ejecuta antes de importar CUDA)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import pandas as pd
from tqdm import tqdm
from autodistill_grounded_sam import GroundedSAM
from autodistill.detection import CaptionOntology

DATASET_PATH = "./datasets/sAFRICANWILDLIFE_FLUXFP8"
df = pd.read_csv(f"{DATASET_PATH}/prompts.csv")
df = df[df["class"] != 'misc']

# Orden fijo de clases en los .txt (YOLO): una entrada por clase del dataset
ALL_CLASSES = sorted(df["class"].unique())
CLASS_TO_ID = {c: i for i, c in enumerate(ALL_CLASSES)}

# Ontología dummy; en el bucle se asigna la ontología por imagen
model = GroundedSAM(ontology=CaptionOntology({"init": "init"}))

print(f"Clases ({len(ALL_CLASSES)}): {ALL_CLASSES}")
print(f"Filas / imágenes: {len(df)}")

trying to load grounding dino directly


torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)


final text_encoder_type: bert-base-uncased


'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


downloading dino model weights
final text_encoder_type: bert-base-uncased


Por cada fila se usa **`row["class"]`** como clase YOLO en el `.txt` (índice según `ALL_CLASSES`). El texto que ve Grounding DINO es `grounding_prompt_for(clase)` (p. ej. varias clases de barco → `"boat"`). No uses una lista fija tipo `ONTOLOGY_CLASSES = ["car"]`: con un solo prompt, `class_id` siempre es `0` y todas las cajas se etiquetaban mal.

In [10]:
import pandas as pd
from tqdm import tqdm
from autodistill_grounded_sam import GroundedSAM
from autodistill.detection import CaptionOntology

classes_dict ={}
# Orden fijo de clases en los .txt (YOLO): una entrada por clase del dataset
ALL_CLASSES = ['tugboat']
CLASS_TO_ID = {
    "tugboat" : 5
}

for x in ALL_CLASSES:
    classes_dict[x] = x






# Clase que guardas en el label → texto que usa GroundingDINO (p. ej. "person" detecta mejor)
GROUNDING_PROMPT_BY_CLASS = {
    "pedestrian": "person",
    "Person_sitting": "person",
    "person_sitting": "person",
    'cruise ship': 'boat',
 'fishing boat': 'boat',
 'human powered boat': 'boat',
 'industrial ship': 'boat',
 'jetski': 'boat',
 'military_ship': 'boat',
 'passenger ship': 'boat',
 'pedal boat': 'boat',
 'sailboat': 'boat',
 'small boat': 'boat',
 'speed_boat': 'boat',
 'tugboat': 'boat'
}

DATASET_PATH = "./datasets/sFERNANDO_TUGBOATS_FLUXDEV"
def grounding_prompt_for(label: str) -> str:
    return GROUNDING_PROMPT_BY_CLASS.get(label, GROUNDING_PROMPT_BY_CLASS.get(label.lower(), label))


In [12]:
import cv2
import supervision as sv
import json
import numpy as np
import pandas as pd
from datetime import datetime
import os

path = DATASET_PATH
output_path = os.path.join(path, "labels")
labeled_images_dir = os.path.join(path, "images_label")
json_dir = os.path.join(path, "detections_json")
os.makedirs(output_path, exist_ok=True)
os.makedirs(labeled_images_dir, exist_ok=True)
os.makedirs(json_dir, exist_ok=True)

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

# Prefer explicit image id from CSV (`Unnamed: 0`) if present — matches `images/{id}.png`
_ID_COL = "Unnamed: 0"
df = pd.read_csv(f"{DATASET_PATH}/prompts.csv")

def row_image_id(index, row) -> int:
    if _ID_COL in row.index:
        return int(row[_ID_COL])
    return int(index)


for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    try:
        img_id = row_image_id(index, row)
        gt_class = str(row["class"])
        grounding_prompt = grounding_prompt_for(gt_class)

        # prompt sent to Grounding DINO → canonical class stored on detections
        model.ontology = CaptionOntology({grounding_prompt: gt_class})

        image_path = os.path.join(path, "images", f"{img_id}.jpg")
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"No se pudo leer: {image_path}")

        h, w = image.shape[:2]
        txt_path = os.path.join(output_path, f"{img_id}.txt")

        detections = model.predict(image_path)
        detections = detections.with_nms()

        yolo_class_id = CLASS_TO_ID[gt_class]

        record = {
            "index": img_id,
            "dataframe_iter_index": int(index) if isinstance(index, (int, np.integer)) else index,
            "image_path": image_path,
            "image_width": w,
            "image_height": h,
            "ground_truth_class": gt_class,
            "grounding_prompt": grounding_prompt,
            "timestamp": datetime.now().isoformat(),
            "num_detections": len(detections) if detections is not None else 0,
            "detections": [],
        }

        # One GT class per image row; every box is written with CLASS_TO_ID[gt_class]
        if len(detections) > 0 and detections.xyxy is not None and len(detections.xyxy) > 0:
            confidences = detections.confidence if detections.confidence is not None else np.zeros(len(detections.xyxy))
            vis_labels = [f"{gt_class} {float(conf):.2f}" for conf in confidences]

            for i, (box, conf) in enumerate(zip(detections.xyxy, confidences)):
                x_min, y_min, x_max, y_max = box
                x_center = ((x_min + x_max) / 2) / w
                y_center = ((y_min + y_max) / 2) / h
                bw = (x_max - x_min) / w
                bh = (y_max - y_min) / h

                record["detections"].append({
                    "detection_id": i,
                    "class_name": gt_class,
                    "yolo_class_id": yolo_class_id,
                    "confidence": float(conf),
                    "bbox_xyxy": [float(x_min), float(y_min), float(x_max), float(y_max)],
                    "bbox_yolo": [float(x_center), float(y_center), float(bw), float(bh)],
                    "bbox_width_px": float(x_max - x_min),
                    "bbox_height_px": float(y_max - y_min),
                })

            with open(os.path.join(json_dir, f"{img_id}.json"), "w") as jf:
                json.dump(record, jf, indent=2)

            annotated_image = box_annotator.annotate(scene=image.copy(), detections=detections)
            annotated_image = label_annotator.annotate(
                scene=annotated_image,
                detections=detections,
                labels=vis_labels,
            )

            with open(txt_path, "w") as f:
                for box in detections.xyxy:
                    x_min, y_min, x_max, y_max = box
                    x_center = ((x_min + x_max) / 2) / w
                    y_center = ((y_min + y_max) / 2) / h
                    bw = (x_max - x_min) / w
                    bh = (y_max - y_min) / h
                    f.write(f"{yolo_class_id} {x_center} {y_center} {bw} {bh}\n")
        else:
            with open(os.path.join(json_dir, f"{img_id}.json"), "w") as jf:
                json.dump(record, jf, indent=2)
            annotated_image = image.copy()
            open(txt_path, "w").close()

        cv2.imwrite(os.path.join(labeled_images_dir, f"{img_id}.png"), annotated_image)

    except Exception as err:
        try:
            _iid = row_image_id(index, row)
        except Exception:
            _iid = None
        print(f"Error index={index} img_id={_iid}: {err}")


Processing:  25%|██▌       | 2319/9216 [00:00<00:00, 11743.26it/s]

Error index=0 img_id=0: name 'model' is not defined
Error index=1 img_id=1: name 'model' is not defined
Error index=2 img_id=2: name 'model' is not defined
Error index=3 img_id=3: name 'model' is not defined
Error index=4 img_id=4: name 'model' is not defined
Error index=5 img_id=5: name 'model' is not defined
Error index=6 img_id=6: name 'model' is not defined
Error index=7 img_id=7: name 'model' is not defined
Error index=8 img_id=8: name 'model' is not defined
Error index=9 img_id=9: name 'model' is not defined
Error index=10 img_id=10: name 'model' is not defined
Error index=11 img_id=11: name 'model' is not defined
Error index=12 img_id=12: name 'model' is not defined
Error index=13 img_id=13: name 'model' is not defined
Error index=14 img_id=14: name 'model' is not defined
Error index=15 img_id=15: name 'model' is not defined
Error index=16 img_id=16: name 'model' is not defined
Error index=17 img_id=17: name 'model' is not defined
Error index=18 img_id=18: name 'model' is not de

Processing:  51%|█████▏    | 4724/9216 [00:00<00:00, 11917.27it/s]

Error index=2767 img_id=2767: name 'model' is not defined
Error index=2768 img_id=2768: name 'model' is not defined
Error index=2769 img_id=2769: name 'model' is not defined
Error index=2770 img_id=2770: name 'model' is not defined
Error index=2771 img_id=2771: name 'model' is not defined
Error index=2772 img_id=2772: name 'model' is not defined
Error index=2773 img_id=2773: name 'model' is not defined
Error index=2774 img_id=2774: name 'model' is not defined
Error index=2775 img_id=2775: name 'model' is not defined
Error index=2776 img_id=2776: name 'model' is not defined
Error index=2777 img_id=2777: name 'model' is not defined
Error index=2778 img_id=2778: name 'model' is not defined
Error index=2779 img_id=2779: name 'model' is not defined
Error index=2780 img_id=2780: name 'model' is not defined
Error index=2781 img_id=2781: name 'model' is not defined
Error index=2782 img_id=2782: name 'model' is not defined
Error index=2783 img_id=2783: name 'model' is not defined
Error index=27

Processing:  78%|███████▊  | 7161/9216 [00:00<00:00, 11972.98it/s]

Error index=5271 img_id=5271: name 'model' is not defined
Error index=5272 img_id=5272: name 'model' is not defined
Error index=5273 img_id=5273: name 'model' is not defined
Error index=5274 img_id=5274: name 'model' is not defined
Error index=5275 img_id=5275: name 'model' is not defined
Error index=5276 img_id=5276: name 'model' is not defined
Error index=5277 img_id=5277: name 'model' is not defined
Error index=5278 img_id=5278: name 'model' is not defined
Error index=5279 img_id=5279: name 'model' is not defined
Error index=5280 img_id=5280: name 'model' is not defined
Error index=5281 img_id=5281: name 'model' is not defined
Error index=5282 img_id=5282: name 'model' is not defined
Error index=5283 img_id=5283: name 'model' is not defined
Error index=5284 img_id=5284: name 'model' is not defined
Error index=5285 img_id=5285: name 'model' is not defined
Error index=5286 img_id=5286: name 'model' is not defined
Error index=5287 img_id=5287: name 'model' is not defined
Error index=52

Processing: 100%|██████████| 9216/9216 [00:00<00:00, 11851.46it/s]

Error index=7892 img_id=7892: name 'model' is not defined
Error index=7893 img_id=7893: name 'model' is not defined
Error index=7894 img_id=7894: name 'model' is not defined
Error index=7895 img_id=7895: name 'model' is not defined
Error index=7896 img_id=7896: name 'model' is not defined
Error index=7897 img_id=7897: name 'model' is not defined
Error index=7898 img_id=7898: name 'model' is not defined
Error index=7899 img_id=7899: name 'model' is not defined
Error index=7900 img_id=7900: name 'model' is not defined
Error index=7901 img_id=7901: name 'model' is not defined
Error index=7902 img_id=7902: name 'model' is not defined
Error index=7903 img_id=7903: name 'model' is not defined
Error index=7904 img_id=7904: name 'model' is not defined
Error index=7905 img_id=7905: name 'model' is not defined
Error index=7906 img_id=7906: name 'model' is not defined
Error index=7907 img_id=7907: name 'model' is not defined
Error index=7908 img_id=7908: name 'model' is not defined
Error index=79